# Initial QC

+ Plate 1: 96 samples(60/36), on MEGA kit v2, seq 15 sub-lib X 3 runs
+ Plate 2: 96 samples (88/8), on MEGA kit v2, seq 16 sub-lib X 3 runs
+ Plate 3: 96 samples (96), on MEGA kit v2, seq 12 sub-lib X 3 runs
+ ~30 Billion reads per plate
+ Parse workflow has a low [doublet rate](https://support.parsebiosciences.com/hc/en-us/articles/360053107311-What-is-the-expected-doublet-rate#:~:text=Doublet%20rates%20are%20low%2C%20less,through%20the%20Whole%20Transcriptome%20workflow.): ~3% per 100K cells
+ Moved to Scanpy from Seurat as combined object size of both plates was too much for R
+ FC samples included
+ Rm genes [MitoCarta 3.0](https://academic.oup.com/nar/article/49/D1/D1541/5974091) genes
  + Downloaded [Human.MitoCarta3.0.xls](https://personal.broadinstitute.org/scalvo/MitoCarta3.0/Human.MitoCarta3.0.xls) from the [Broad Institute site](https://www.broadinstitute.org/files/shared/metabolism/mitocarta/human.mitocarta3.0.html) 

+ TODO:
  + Maybe add some plots extra QC plots
  + Code folding to get rid of large outputs

In [ ]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from scanpy_init_env import *
from scanpy_utils import *
from scanpy_gene_lists import *

# Set variables and filters
rm_doublets = True
rm_mad_outliers = False
genes_per_cell_thresh = {'min': 1000, 'max': 5000}
counts_per_cell_thresh = {'min': 500, 'max': None}
mito_ribo_thresh = {'mito': 5, 'ribo': 5}
cells_per_gene_thresh = 5
cells_per_smpl_thresh = 10

### Filters

In [ ]:
thresholds = [
    {'Threshold': 'Genes per Cell', 'Limit': f"{genes_per_cell_thresh['min']}–{genes_per_cell_thresh['max']}"},
    {'Threshold': 'Counts per Cell', 'Limit': f"≥{counts_per_cell_thresh['min']}"},
    {'Threshold': 'Mitochondrial Reads', 'Limit': f"≤{mito_ribo_thresh['mito']}%"},
    {'Threshold': 'Ribosomal Reads', 'Limit': f"≤{mito_ribo_thresh['ribo']}%"},
    {'Threshold': 'Cells per Gene', 'Limit': f"≥{cells_per_gene_thresh}"}
]
thresholds_df = pd.DataFrame(thresholds)

### Load data

In [ ]:
# Initialize the environment and get all paths and logger
# Loading for h5ad takes a few mins, from mtx takes 50 mins locally
#%pip install xlrd # Run once on Hawk
logger, root_dir, sheets_dir, plate_dir, scanpy_dir, report_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = load_and_dwnsmpl_data(None, False, plate_dir)
mito_genes = pd.read_excel(sheets_dir + 'Human.MitoCarta3.0.xls', sheet_name = 'A Human MitoCarta3.0')['Symbol'].tolist()
adata.obs['sample'] = adata.obs['sample'].str.replace('sample_', '')
adata = adata[~adata.obs['sample'].str.endswith(tuple(['WGE', 'Hipp', 'Thal']))]

# Save sample counts to TSV before filtering / sample removal
sample_counts = adata.obs['sample'].value_counts().reset_index(name='count').assign(plate=plate)
sample_counts.to_csv(f"{report_dir}/sample_counts_{plate}.tsv", sep='\t', index=False)

adata.obs['sublibrary'] = [x[1] for x in adata.obs.index.str.split('__s')] 

In [ ]:
# Check if matrix contains only integers: Move this to loading function?
subset = adata.X[:1000, :1000]
is_all_integers = np.all(np.mod(subset.data, 1) == 0)
print(f'Are all values of adata.X integers? {is_all_integers}')

### Initial counts

In [ ]:
# Dimensions
logger.info(f"Original dimensions (cells x genes): {adata.shape}")

In [ ]:
# Cell counts by sample
logger.info(f"Number of samples: {adata.obs['sample'].nunique()}")
adata.obs['sample'].value_counts()

In [ ]:
# Check gene IDs
adata.var

# QC metadata

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(adata, qc_vars = ["mt", "ribo", "hb"], inplace = True, log1p = True)
adata.obs

In [ ]:
# Check if tscp_count = total_counts 
# Parse filtered h5ad file has non integer values in total counts so use Parse mtx file to load data in instead
adata.obs[['tscp_count', 'total_counts']] 

In [ ]:
sample_qc_metrics = (
    adata.obs.groupby("sample")
    .agg(
        median_genes_per_cell = ("n_genes_by_counts", "median"),
        median_counts_per_cell = ("total_counts", "median"),
        total_cells = ("n_genes_by_counts", "count"),
    )
    .reset_index()
)
sample_qc_metrics

In [ ]:
# Most experessed genes - Not such a big deal now?
logger.info("Checking most exp genes ...")
sc.pl.highest_expr_genes(adata, n_top=20)

### Distributions per-sample pre-filter

In [ ]:
# Create the violin plot
fig, ax = plt.subplots(figsize=(10, 6))
sc.pl.violin(
    adata,
    keys='gene_count',
    jitter=0.4,
    groupby='sample',
    rotation=90,
    size=1.5,
    ax=ax,
    show=False, 
    color='Red'
)
plt.title('Genes per cell - pre-filtering')
plt.axhline(y = genes_per_cell_thresh['min'], color = 'r', linestyle = '-')
plt.axhline(y = genes_per_cell_thresh['min'], color = 'r', linestyle = '-') 

fig, ax = plt.subplots(figsize=(10, 6))
sc.pl.violin(
    adata,
    keys='total_counts',
    jitter=0.4,
    groupby='sample',
    rotation=90,
    size=1.5,
    ax=ax,
    show=False, 
    color='Red'
)
plt.title('Counts per cell - pre-filtering')
plt.axhline(y = counts_per_cell_thresh['min'], color = 'r', linestyle = '-')

### Filtering

In [ ]:
# Filter 1
logger.info("Applying filter 1 ...")
logger.info("\n%s", thresholds_df.to_string())
filter_anndata(
    adata, 
    min_genes_per_cell = genes_per_cell_thresh['min'], 
    max_genes_per_cell = genes_per_cell_thresh['max'],
    min_counts_per_cell = counts_per_cell_thresh['min'], 
    min_cells_per_gene = cells_per_gene_thresh, 
    min_cells_per_sample = cells_per_smpl_thresh)

In [ ]:
# Doublets
if "counts" not in adata.layers:
    print("Saving raw counts ...")
    adata.layers["counts"] = adata.X.copy()  # Store raw counts in adata.layers
if rm_doublets is True:
    logger.info("Detecting Doublets ...")
    sc.external.pp.scrublet(adata, batch_key = 'sublibrary') # Need to subset by sublibrary or Scrublet fails on Hawk 

In [ ]:
# Check if matrix contains only integers
adata.layers["counts"][0:10, 0:10].todense()

In [ ]:
# Code for Scrublet
# Make sure to save raw counts before normalisation
if rm_doublets is True:
    logger.info("Plotting doublets ...")   
    plot_doublet_umaps(adata)
adata.X = adata.layers["counts"].copy()  # Restore raw counts
adata.X[:10,:10].todense()  # Remove

In [ ]:
logger.info("Applying filter 2 ...")
filter_cells_and_genes(
    adata,
    mito_threshold = mito_ribo_thresh['mito'],
    ribo_threshold = mito_ribo_thresh['ribo'],
    remove_doublets = rm_doublets,
    remove_mad_outliers = rm_mad_outliers,
    genes_to_remove = [mito_genes, 'MALAT1']  # New parameter
)

### Final Counts

In [ ]:
# Final Dimensions
logger.info(f"Final dimensions (cells x genes): {adata.shape}") 

In [ ]:
# Cells per sample post-filter
adata.obs['sample'].value_counts()

### Final distributions per sample

In [ ]:
# Create the violin plot
fig, ax = plt.subplots(figsize=(10, 6))
sc.pl.violin(
    adata,
    keys='gene_count',
    jitter=0.4,
    groupby='sample',
    rotation=90,
    size=1.5,
    ax=ax,
    show=False, 
    color='Red'
)
plt.title('Final gene per cell')
plt.axhline(y = genes_per_cell_thresh['min'], color = 'r', linestyle = '-')
plt.axhline(y = genes_per_cell_thresh['min'], color = 'r', linestyle = '-') 

fig, ax = plt.subplots(figsize=(10, 6))
sc.pl.violin(
    adata,
    keys='total_counts',
    jitter=0.4,
    groupby='sample',
    rotation=90,
    size=1.5,
    ax=ax,
    show=False, 
    color='Red'
)
plt.title('Final counts per cell')
plt.axhline(y = counts_per_cell_thresh['min'], color = 'r', linestyle = '-')

In [ ]:
# Save
logger.info("Saving h5ad file ...")
adata.X = adata.layers["counts"].copy() # Restore raw counts for saving as merging in next script
subset = adata.X[:1000, :1000]
is_all_integers = np.all(np.mod(subset.data, 1) == 0)
print(f'Are all values of adata.X integers? {is_all_integers}')
adata.write(scanpy_dir + f'adata_qc_{plate}.h5ad')
logger.info(f"{plate} qc run done.")